# Notebook 03: Feature Engineering & Domain Transformation

**Project**: Telecom Customer Churn Prediction & Retention System
**Task**: TASK 06 - Feature Engineering & Domain Transformation

---

In [ ]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from src.features import FeatureEngineeringTransformer, engineer_features
from src.utils import get_project_root

project_root = get_project_root()


## 1. Load Cleaned Dataset

In [ ]:
cleaned_path = project_root / "data" / "processed" / "cleaned_churn_data.csv"
df_clean = pd.read_csv(cleaned_path)
print(f"Cleaned Dataset: {df_clean.shape[0]} rows x {df_clean.shape[1]} cols")
df_clean.head(3)


## 2. Run Feature Engineering Pipeline

In [ ]:
df_eng = engineer_features(cleaned_df_path=cleaned_path, save_processed=True)
print(f"Engineered Dataset: {df_eng.shape[0]} rows x {df_eng.shape[1]} cols")
print(f"Total predictor features: {df_eng.shape[1] - 2}  (excluding customerID and Churn)")
print(f"Total nulls: {df_eng.isnull().sum().sum()}")
df_eng.head(3)


## 3. Engineered Feature Verification

In [ ]:
# 3A: Show all features by category
REQUIRED_DOMAIN = ['AvgMonthlyChargeRatio', 'MonthlyToTotalRatio', 'TotalServicesCount',
                    'HasTechSupportSecurity', 'IsAutomaticPayment']
print("=== REQUIRED Domain Features ===")
print(df_eng[REQUIRED_DOMAIN].describe().T.round(4))

print()
print("=== Tenure Cohort Binning (One-Hot) ===")
cohort_cols = [c for c in df_eng.columns if 'TenureCohort' in c]
print(df_eng[cohort_cols].sum().to_dict())

print()
print("=== One-Hot Encoded Nominal Features ===")
ohe_cols = [c for c in df_eng.columns if '_' in c and c not in REQUIRED_DOMAIN + cohort_cols]
print(ohe_cols)


## 4. Feature Engineering Decisions & Observations

1. **Tenure Cohort (`TenureCohort`)**: Binned `tenure` into 4 lifecycle groups (`0-12m`, `12-24m`, `24-48m`, `48-72m`). One-hot encoded with `0-12m` as reference. Captures non-linear tenure lifecycle churn dynamics.

2. **Avg Monthly Charge Ratio (`AvgMonthlyChargeRatio = TotalCharges / (tenure + 1)`)**: Detects whether a customer's historical billing is consistent with current charges. Useful for identifying recent price hikes.

3. **Monthly-to-Total Ratio (`MonthlyToTotalRatio = MonthlyCharges / (TotalCharges + 1)`)**: High values indicate short-tenure customers spending proportionally more per month relative to lifetime.

4. **Total Services Count (`TotalServicesCount`)**: Ecosystem bundling index. Higher bundle count correlates with higher switching costs and lower churn probability.

5. **Tech Support + Security Flag (`HasTechSupportSecurity`)**: Binary indicator. Customers with both premium add-ons show <15% churn rate.

6. **Auto-Payment Indicator (`IsAutomaticPayment`)**: Automatic billing reduces monthly payment friction and correlates with lower churn.

7. **Categorical Encodings**: Binary features mapped 0/1; multi-class nominals one-hot encoded with `drop_first=True` to avoid dummy variable trap.

8. **StandardScaler**: Applied to 6 continuous features fitted strictly on training data to prevent data leakage during modeling.

## 5. Feature Selection (TASK 07)

We utilize Mutual Information, Random Forest Feature Importances, and Recursive Feature Elimination with Cross-Validation (RFECV) to select the most impactful features.

In [ ]:
from src.feature_selection import FeatureSelector
X = df_eng.drop(columns=['Churn', 'customerID'])
y = df_eng['Churn']

selector = FeatureSelector()
importance_df, selected_features = selector.rank_and_select_features(X, y)

print(f"RFECV selected {len(selected_features)} features out of {X.shape[1]}")


### 5.1 Top 10 Features (Combined Ranking)

In [ ]:
importance_df.head(10)

### 5.2 Eliminated Features

In [ ]:
eliminated = importance_df[~importance_df['RFECV_Selected']]
print(f"Total Eliminated Features: {len(eliminated)}")
eliminated


### 5.3 Pruned Dataset Shape

In [ ]:
cols_to_keep = ['customerID'] + selected_features + ['Churn']
df_selected = df_eng[cols_to_keep]
print("Pruned Dataset Shape:", df_selected.shape)
df_selected.head(3)
